In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
N = 100
h = 1
base_d = 0.02
base_w = base_d * 6
base_triArea = 0.001
numSegments = 100
tilt_n = 20
total_width = base_d + base_w
clipping_lN = 50
clipping_uN = 53

In [ ]:
N = 5
clipping_lN = None
clipping_uN = None

In [ ]:
freq = 1
amplitude = 1

In [ ]:
from ipywidgets import interactive, widgets
def plotForAlpha(freq, amplitude, tilt_n): visualization.plot_line_segments(*parametric_pillows.sinusoid_raw(N = N, h = h, d = base_d / freq, w = base_w / freq, triArea = base_triArea / freq**2, numSegments = int(numSegments * freq * 0.5), freq = np.pi * freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN))
iplot = interactive(plotForAlpha, 
                    freq = widgets.FloatSlider(min=1, max=10, value=1, step=1), 
                    amplitude = widgets.FloatSlider(min=0.01, max=2, value=0.1, step=0.01), 
                    tilt_n = widgets.IntSlider(min=1, max=30, value=1, step=1))
iplot.children[-1].layout.height = '500px'
display(iplot)

In [ ]:
freq = iplot.children[0].value
amplitude = iplot.children[1].value
tilt_n = iplot.children[2].value

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
pts, edges = parametric_pillows.sinusoid_raw(N = N, h = h, d = base_d / freq, w = base_w / freq, triArea = base_triArea / freq**2, numSegments = int(numSegments * freq * 0.5), freq = np.pi * freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN, target_length = base_d / freq)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
with open("point.obj", 'w') as f:
    for pt in pts:
        f.write("v {} {} {}\n".format(pt[0], pt[1], 0))

In [ ]:
plt.figure(figsize = (5, 5))
plt.scatter(np.array(pts)[:, 0], np.array(pts)[:, 1])
plt.axis("equal")
plt.savefig('pattern.png')

In [ ]:
import sheet_meshing

In [ ]:
importlib.reload(sheet_meshing)

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
m, fuseMarkers, brdyWallMarkers = parametric_pillows.sinusoid(N = N, h = h, d = base_d / freq, w = base_w / freq, triArea = base_triArea / freq**2 * base_w * 0.5, numSegments = int(numSegments * freq * 0.5), freq = np.pi * freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN, meshingOption = "Y", target_length = base_d / freq, mark_last_segment_as_wall=True)

visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
m, fuseMarkers, brdyWallMarkers = parametric_pillows.sinusoid(N = N, h = h, d = 1e-16, w = base_w / freq, triArea = base_triArea / freq**2 * base_w * 0.5, numSegments = int(numSegments * freq * 0.5), freq = np.pi * freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN, meshingOption = "Y", target_length = base_d / freq * 0.5, mark_last_segment_as_wall=True, remove_end_walls = True)

                
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
isheet.rigidMotionPinVars

In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 50 / freq
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts, callback=cb)
benchmark.report()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])

In [ ]:
viewer.update()

### Repeat the inflation, this time recording it to a video
Requires `MeshFEM`'s `OffscreenRenderer` to be successfully built.

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

from tri_mesh_viewer import OffscreenTriMeshViewer
oview = OffscreenTriMeshViewer(isheet, width=768, height=640, wireframe=True)

benchmark.reset()
opts.niter=1000
oview.recordStart('cc_inflate.mp4')
isheet.pressure = 20 * 3.75
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts,
                                callback=lambda it: oview.update())
benchmark.report()
oview.recordStop()

In [ ]:
# Some basic statistics for the deformation
from matplotlib import pyplot as plt
strains = utils.getStrains(isheet)[:, 0]
plt.hist(strains, 60);
plt.xlabel('Principal stretch $\\lambda_0$')
print(np.median(strains))

### Analyze curvature of the inflated surface

In [ ]:
isa = inflation.InflatedSurfaceAnalysis(isheet)
curvature = isa.curvature()
metric = isa.metric()

In [ ]:
import matplotlib, vis
from tri_mesh_viewer import TriMeshViewer
isurf = isa.inflatedSurface()
metric_vf = vis.fields.VectorField(isurf, metric.sigma_2[:, None] * metric.left_stretch, vmin=0, vmax=1.0,
                                   align=vis.fields.VectorAlignment.CENTER, colormap=matplotlib.cm.viridis,
                                   glyph=vis.fields.VectorGlyph.CYLINDER)

viewer2 = TriMeshViewer(isurf, width=768, height=640, scalarField=vis.fields.ScalarField(isurf, curvature.meanCurvature(), colormap=matplotlib.cm.coolwarm), vectorField=metric_vf)
viewer2.showWireframe()
viewer2.show()